# Crogrammar v0.2.0 — trening na Kaggle (ByT5-base)

Veliki upgrade: ByT5-base, hr500k + hrvatska Wikipedia, prošireni generator grešaka,
RAPUT stvarne greške u trening (pretežane), 3 epohe. Preporuka: **GPU P100**, fp32.

Storage: sve u `/kaggle/working` (nema Google Drive-a). Za nastavak treninga preko
sesija spremi output kao Kaggle Dataset i dodaj ga kao input u sljedećem run-u.

## 1. Instalacija

In [ ]:
!git clone https://github.com/mmlinaric/crogrammar.git
%cd crogrammar
!pip install -e ".[train]"

## 2. Preuzimanje izvora (hr500k, hunspell-hr, RAPUT)

In [ ]:
from crogrammar.data.download import download_hr500k, download_hunspell_dic, download_raput
download_hr500k("data/raw")
download_hunspell_dic("data/raw")
raput_path = download_raput("data/raw")
print("raput:", raput_path)

## 3. Čist tekst: hr500k + hrvatska Wikipedia (~1M rečenica)

In [ ]:
import gzip
from pathlib import Path
from crogrammar.data.conllu import parse_conllu, sentence_text
from crogrammar.data.wikipedia import load_wiki_sentences

clean = []
for gz in Path("data/raw/hr500k").glob("*.conllu.gz"):
    with gzip.open(gz, "rt", encoding="utf-8") as f:
        for sent in parse_conllu(f.read()):
            clean.append(sentence_text(sent))
print("hr500k recenica:", len(clean))

wiki = list(load_wiki_sentences(limit=1_000_000))
print("wiki recenica:", len(wiki))
clean += wiki
print("ukupno cistih:", len(clean))

## 4. Izgradnja dataseta (sintetika + RAPUT stvarni, pretežani x4)

In [ ]:
from crogrammar.data.confusion import load_confusion_from_dic, load_wordset_from_dic
from crogrammar.data.build_dataset import make_pairs, mix_sources, split_pairs, write_jsonl
from crogrammar.data.raput import raput_training_pairs

confusion = load_confusion_from_dic("data/raw/hr_HR.dic")
real_words = load_wordset_from_dic("data/raw/hr_HR.dic")
synthetic = make_pairs(clean, confusion, seed=42, variants=2, real_words=real_words)
print("sintetickih parova:", len(synthetic))

real = raput_training_pairs(str(raput_path))
print("RAPUT stvarnih parova:", len(real))

all_pairs = mix_sources(synthetic, real, real_weight=4, seed=42)
train, dev, test = split_pairs(all_pairs, dev_frac=0.02, test_frac=0.02, seed=42)
write_jsonl(train, "data/processed/train.jsonl")
write_jsonl(dev, "data/processed/dev.jsonl")
write_jsonl(test, "data/processed/test.jsonl")
print(len(train), len(dev), len(test))

## 5. Trening (ByT5-base, fp32, 3 epohe, resume-safe)

Ako je prethodni run spremio checkpoint kao Kaggle Dataset, dodaj ga kao input
(npr. `/kaggle/input/crogrammar-ckpt`) i kopiraj u `output_dir` prije pokretanja:
`!cp -r /kaggle/input/crogrammar-ckpt/byt5-hr-gec /kaggle/working/`. Auto-detekcija
checkpointa u `train()` tada nastavlja od zadnjeg.

In [ ]:
from crogrammar.train.config import TrainConfig
from crogrammar.train.train import train

cfg = TrainConfig(
    base_model="google/byt5-base",
    output_dir="/kaggle/working/byt5-hr-gec",
    batch_size=4,
    grad_accum=8,
    max_source_len=160,
    max_target_len=160,
    num_epochs=3,
    fp16=False,
)
train(cfg, resume_from_checkpoint=True)

## 6. Evaluacija na ručnom test setu (GLEU ukupno + po kategoriji)

In [ ]:
import json
from collections import defaultdict
from crogrammar.inference.gec import CroatianGEC
from crogrammar.eval.run_eval import evaluate_pairs_batched

gec = CroatianGEC(model_path="/kaggle/working/byt5-hr-gec")

with open("tests/manual_test_set.jsonl", encoding="utf-8") as f:
    manual = [json.loads(l) for l in f if l.strip()]

overall = evaluate_pairs_batched(manual, gec.generate_batch, batch_size=32)
print("GLEU (rucni set):", round(overall["gleu"], 4), "| n:", overall["n"])

by_cat = defaultdict(list)
for r in manual:
    by_cat[r["kategorija"]].append(r)
for cat, rows in sorted(by_cat.items()):
    s = evaluate_pairs_batched(rows, gec.generate_batch, batch_size=32)
    print(f"  {cat:16s}: GLEU {round(s['gleu'],4)} (n={s['n']})")

## 7. METADATA + priprema za spremanje kao Kaggle Dataset

In [ ]:
import json, datetime, os
meta = {
    "version": "0.2.0",
    "date": datetime.date.today().isoformat(),
    "base_model": "google/byt5-base",
    "gleu_manual": round(overall["gleu"], 4),
    "license": "CC BY-SA 4.0",
    "sources": ["hr500k", "hunspell-hr", "RAPUT (trening)", "Wikipedia hr"],
}
os.makedirs("/kaggle/working/byt5-hr-gec", exist_ok=True)
with open("/kaggle/working/byt5-hr-gec/METADATA.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(meta)
print("Model je u /kaggle/working/byt5-hr-gec -> 'Save Version' cuva output; za resume kreiraj Kaggle Dataset od te mape.")